# Logs

> Generating / formatting the base logging instance

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Annotated
from enum import Enum

import os, sys, json
from pathlib import Path

import typer
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import print as rprint

from mcp.server.fastmcp import FastMCP

from nbdev_mcp import __version__
from nbdev_mcp.utils.logs import log
from nbdev_mcp.utils.config import CURRENT_PROJECT
from nbdev_mcp.utils.paths import resolve_selector
from nbdev_mcp.utils.subprocess import watch_notebooks
from nbdev_mcp.resources import add_resources
from nbdev_mcp.tools import (
    add_project_tools, add_env_tools, add_nbdev_tools, 
    add_notebook_editing_tools, add_lint_tools, 
    add_analysis_tools, add_tests_tools
)
from nbdev_mcp.prompts import add_prompts
from nbdev_mcp.tasks import add_task_tools, enable_nbdev_tasks

console = Console()

## HTTP Utils

In [ ]:
#| export
def set_http_path_if_supported(target_path: str) -> bool:
    """Try to set the HTTP mount path on the MCP SDK settings.
    
    Parameters
    ----------
    target_path : str
        The path to set (e.g., '/mcp').
    
    Returns
    -------
    bool
        True if successfully set, False if not supported.
    """
    import mcp
    try:
        mcp.settings.streamable_http_path = target_path  # type: ignore[attr-defined]
        return True
    except Exception:
        try:
            mcp.settings.http_path = target_path  # type: ignore[attr-defined]
            return True
        except Exception:
            return False

## Create the MCP

In [ ]:
#| export
def create_nbdev_mcp(name: str = "mcp.nbdev") -> FastMCP:
    """Create and configure the nbdev MCP server with all resources, tools, and prompts."""
    mcp = FastMCP(name)
    
    # Enable experimental tasks API
    enable_nbdev_tasks(mcp)
    
    # Attach all nbdev-related resources, tools, and prompts
    add_resources(mcp)
    add_project_tools(mcp)
    add_env_tools(mcp)
    add_nbdev_tools(mcp)
    add_notebook_editing_tools(mcp)  # CRITICAL: Notebook editing and workflow tools
    add_prompts(mcp)  # Philosophy prompts must come after tools are registered
    
    # Extensions: linting, analysis, and code generation tools
    add_lint_tools(mcp)
    add_analysis_tools(mcp)
    add_tests_tools(mcp)
    
    # Task-based tools for auditing, deduplication, and refactoring
    add_task_tools(mcp)
    
    return mcp

## `main`

In [ ]:
#| export
class Transport(str, Enum):
    """MCP transport modes."""
    stdio = "stdio"
    http = "http"
    streamable_http = "streamable-http"


class Provider(str, Enum):
    """Supported MCP client providers."""
    claude = "claude"
    vscode = "vscode"
    cursor = "cursor"


app = typer.Typer(
    name="nbdev-mcp",
    help="[bold blue]nbdev MCP server[/] - Model Context Protocol for nbdev projects",
    rich_markup_mode="rich",
    no_args_is_help=False,
    add_completion=True,
    context_settings={"help_option_names": ["-h", "--help"]},
)


def version_callback(value: bool):
    if value:
        rprint(f"[bold blue]nbdev-mcp[/] [dim]v{__version__}[/]")
        raise typer.Exit()


@app.callback(invoke_without_command=True)
def callback(
    ctx: typer.Context,
    version: Annotated[bool, typer.Option(
        "-V", "--version", callback=version_callback, is_eager=True,
        help="Show version and exit."
    )] = False,
    # Default run options (when no subcommand given)
    project: Annotated[Optional[str], typer.Option(
        "-p", "--project", help="Path or alias for an nbdev project."
    )] = None,
    transport: Annotated[Transport, typer.Option(
        "-t", "--transport", help="Transport mode.",
        envvar="NBDEV_MCP_TRANSPORT"
    )] = Transport.stdio,
    verbose: Annotated[bool, typer.Option(
        "-v", "--verbose", help="Enable debug output."
    )] = False,
):
    """Run the MCP server (default command)."""
    if ctx.invoked_subcommand is None:
        _run_server(project=project, transport=transport, verbose=verbose)

## Next

In [ ]:
#| export


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Install helpers

In [ ]:
#| export
def get_claude_config_path() -> Path:
    """Get Claude Desktop config path."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Claude/claude_desktop_config.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Claude/claude_desktop_config.json"
    else:
        return Path.home() / ".config/Claude/claude_desktop_config.json"


def get_vscode_config_path() -> Path:
    """Get VS Code MCP settings path."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Code/User/settings.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Code/User/settings.json"
    else:
        return Path.home() / ".config/Code/User/settings.json"


def get_cursor_config_path() -> Path:
    """Get Cursor MCP settings path."""
    if sys.platform == "darwin":
        return Path.home() / "Library/Application Support/Cursor/User/settings.json"
    elif sys.platform == "win32":
        return Path(os.environ.get("APPDATA", "")) / "Cursor/User/settings.json"
    else:
        return Path.home() / ".config/Cursor/User/settings.json"


def get_config_path(provider: Provider) -> Path:
    """Get config path for a provider."""
    match provider:
        case Provider.claude:
            return get_claude_config_path()
        case Provider.vscode:
            return get_vscode_config_path()
        case Provider.cursor:
            return get_cursor_config_path()


def get_python_path() -> str:
    """Get current Python interpreter path."""
    return sys.executable


def make_server_config() -> dict:
    """Generate server configuration dict."""
    return {
        "command": get_python_path(),
        "args": ["-m", "nbdev_mcp"]
    }


def _parse_jsonc(text: str) -> dict:
    """Parse JSON with Comments (JSONC) - strips // and /* */ comments."""
    import re
    # Remove single-line comments
    text = re.sub(r'//.*?$', '', text, flags=re.MULTILINE)
    # Remove multi-line comments
    text = re.sub(r'/\*.*?\*/', '', text, flags=re.DOTALL)
    # Remove trailing commas before } or ]
    text = re.sub(r',(\s*[}\]])', r'\1', text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {}


def install_to_provider(provider: Provider, dry_run: bool = False) -> bool:
    """Install nbdev-mcp to a provider's config."""
    config_path = get_config_path(provider)
    server_config = make_server_config()
    
    if dry_run:
        console.print(Panel.fit(
            f"[dim]Provider:[/] {provider.value}\n"
            f"[dim]Config:[/] {config_path}\n"
            f"[dim]Python:[/] {server_config['command']}",
            title="[yellow]Dry Run[/]",
            border_style="yellow"
        ))
        return True
    
    # Read or create config
    if config_path.exists():
        config = _parse_jsonc(config_path.read_text())
    else:
        config_path.parent.mkdir(parents=True, exist_ok=True)
        config = {}
    
    # Provider-specific key
    if provider == Provider.claude:
        if "mcpServers" not in config:
            config["mcpServers"] = {}
        config["mcpServers"]["nbdev"] = server_config
    else:  # vscode, cursor use mcp.servers
        if "mcp.servers" not in config:
            config["mcp.servers"] = {}
        config["mcp.servers"]["nbdev"] = server_config
    
    config_path.write_text(json.dumps(config, indent=2))
    console.print(f"[green]✓[/] Installed to [bold]{provider.value}[/]: {config_path}")
    return True


def uninstall_from_provider(provider: Provider, dry_run: bool = False) -> bool:
    """Remove nbdev-mcp from a provider's config."""
    config_path = get_config_path(provider)
    
    if not config_path.exists():
        console.print(f"[yellow]⚠[/] {provider.value}: Config not found")
        return False
    
    config = _parse_jsonc(config_path.read_text())
    
    # Provider-specific key
    key = "mcpServers" if provider == Provider.claude else "mcp.servers"
    
    if key not in config or "nbdev" not in config.get(key, {}):
        console.print(f"[yellow]⚠[/] {provider.value}: nbdev-mcp not installed")
        return False
    
    if dry_run:
        console.print(f"[yellow]Would remove[/] 'nbdev' from {provider.value}")
        return True
    
    del config[key]["nbdev"]
    config_path.write_text(json.dumps(config, indent=2))
    console.print(f"[green]✓[/] Removed from [bold]{provider.value}[/]")
    return True


def check_provider_status(provider: Provider) -> dict:
    """Check installation status for a provider."""
    config_path = get_config_path(provider)
    
    if not config_path.exists():
        return {"provider": provider.value, "installed": False, "exists": False, "path": str(config_path)}
    
    try:
        config = _parse_jsonc(config_path.read_text())
    except Exception:
        return {"provider": provider.value, "installed": False, "exists": True, "path": str(config_path), "error": "parse error"}
    
    key = "mcpServers" if provider == Provider.claude else "mcp.servers"
    installed = key in config and "nbdev" in config.get(key, {})
    
    return {"provider": provider.value, "installed": installed, "exists": True, "path": str(config_path)}

In [ ]:
#| export
def _run_server(
    project: Optional[str] = None,
    transport: Transport = Transport.stdio,
    host: str = "127.0.0.1",
    port: int = 8000,
    path: str = "/mcp",
    verbose: bool = False,
    watch: bool = False,
    watch_interval: float = 2.0,
    watch_cmd: str = "nbdev_export",
) -> None:
    """Internal: run the MCP server."""
    if verbose:
        import logging
        logging.basicConfig(level=logging.DEBUG, format="%(message)s")
        log.setLevel(logging.DEBUG)
        console.print(f"[dim]nbdev-mcp v{__version__}[/]")

    # Set project
    proj_path = None
    if project:
        try:
            proj_path = resolve_selector(project)
            if verbose:
                console.print(f"[green]✓[/] Project: {proj_path}")
        except Exception as e:
            console.print(f"[red]✗[/] {e}")
            if watch:
                raise typer.Exit(1)

    # Watch mode
    if watch:
        if not proj_path:
            console.print("[red]✗[/] Watch mode requires --project")
            raise typer.Exit(1)
        watch_notebooks(proj_path, interval=watch_interval, on_change=watch_cmd)
        return

    # Build MCP
    mcp = create_nbdev_mcp()
    
    match transport:
        case Transport.stdio:
            mcp.run(transport="stdio")
        case Transport.streamable_http | Transport.http:
            default = (host == "127.0.0.1" and port == 8000 and path == "/mcp")
            if default and transport == Transport.streamable_http:
                mcp.run(transport="streamable-http")
            else:
                try:
                    import uvicorn
                except ImportError:
                    console.print("[red]✗[/] uvicorn required: [dim]pip install uvicorn[/]")
                    raise typer.Exit(1)
                if path != "/mcp":
                    set_http_path_if_supported(path)
                uvicorn.run(mcp.streamable_http_app(), host=host, port=port)


@app.command()
def run(
    project: Annotated[Optional[str], typer.Option(
        "-p", "--project", help="Path or alias for nbdev project."
    )] = None,
    transport: Annotated[Transport, typer.Option(
        "-t", "--transport", help="Transport: stdio, http, streamable-http.",
        envvar="NBDEV_MCP_TRANSPORT"
    )] = Transport.stdio,
    host: Annotated[str, typer.Option(
        "-H", "--host", help="Host for HTTP transport.",
        envvar="NBDEV_MCP_HOST"
    )] = "127.0.0.1",
    port: Annotated[int, typer.Option(
        "-P", "--port", help="Port for HTTP transport.",
        envvar="NBDEV_MCP_PORT"
    )] = 8000,
    path: Annotated[str, typer.Option(
        "--path", help="URL path for HTTP.",
        envvar="NBDEV_MCP_PATH"
    )] = "/mcp",
    verbose: Annotated[bool, typer.Option(
        "-v", "--verbose", help="Debug output."
    )] = False,
    watch: Annotated[bool, typer.Option(
        "-w", "--watch", help="Watch notebooks and auto-export."
    )] = False,
    watch_interval: Annotated[float, typer.Option(
        "--watch-interval", help="Watch polling interval (seconds)."
    )] = 2.0,
    watch_cmd: Annotated[str, typer.Option(
        "--watch-cmd", help="Command on change."
    )] = "nbdev_export",
):
    """Run the MCP server."""
    _run_server(
        project=project, transport=transport, host=host, port=port,
        path=path, verbose=verbose, watch=watch,
        watch_interval=watch_interval, watch_cmd=watch_cmd
    )

In [ ]:
#| export
@app.command()
def install(
    provider: Annotated[Optional[Provider], typer.Argument(
        help="Provider to install to (claude, vscode, cursor). Omit for all."
    )] = None,
    dry_run: Annotated[bool, typer.Option(
        "-n", "--dry-run", help="Show what would be done."
    )] = False,
):
    """Install nbdev-mcp to MCP client(s)."""
    providers = [provider] if provider else list(Provider)
    
    for p in providers:
        install_to_provider(p, dry_run=dry_run)
    
    if not dry_run:
        console.print("\n[yellow]Restart your MCP client(s) to activate.[/]")


@app.command()
def uninstall(
    provider: Annotated[Optional[Provider], typer.Argument(
        help="Provider to uninstall from. Omit for all."
    )] = None,
    dry_run: Annotated[bool, typer.Option(
        "-n", "--dry-run", help="Show what would be done."
    )] = False,
):
    """Remove nbdev-mcp from MCP client(s)."""
    providers = [provider] if provider else list(Provider)
    
    for p in providers:
        uninstall_from_provider(p, dry_run=dry_run)


@app.command()
def status():
    """Show installation status for all providers."""
    table = Table(title="nbdev-mcp Status", show_header=True)
    table.add_column("Provider", style="cyan")
    table.add_column("Status")
    table.add_column("Path", style="dim")
    
    for provider in Provider:
        info = check_provider_status(provider)
        if not info["exists"]:
            status = "[dim]Config not found[/]"
        elif info["installed"]:
            status = "[green]✓ Installed[/]"
        else:
            status = "[yellow]Not configured[/]"
        table.add_row(provider.value.title(), status, info["path"])
    
    table.add_row("", "", "")
    table.add_row("Python", f"[green]v{sys.version.split()[0]}[/]", get_python_path())
    table.add_row("nbdev-mcp", f"[blue]v{__version__}[/]", "")
    
    console.print(table)

In [ ]:
#| export
def main():
    """Entry point for console script."""
    app()